In [ ]:
!nvidia-smi

In [ ]:
import torch
print("GPU:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install diffusers transformers accelerate moviepy gTTS sentencepiece huggingface_hub opencv-python scipy

In [ ]:
!ffmpeg -version

In [ ]:
import os

BASE="/kaggle/working/ai_video_factory"

folders=[
"scenes",
"clips",
"audio",
"music",
"video"
]

os.makedirs(BASE, exist_ok=True)

for f in folders:
    os.makedirs(f"{BASE}/{f}", exist_ok=True)

BASE

In [ ]:
import os

def find_transcript():

    search_roots = [
        "/kaggle/working",
        "/kaggle/input",
        "/kaggle"
    ]

    for root in search_roots:
        for dirpath, dirnames, filenames in os.walk(root):
            print("searching in ", dirpath)
            if "transcript.txt" in filenames:
                print("found the file in ", dirpath)
                return os.path.join(dirpath, "transcript.txt")

    return None


INPUT_PATH = find_transcript()

if INPUT_PATH is None:
    raise Exception("transcript.txt not found anywhere in Kaggle runtime")

print("Transcript found at:", INPUT_PATH)

with open(INPUT_PATH, "r") as f:
    transcript = f.read()

print("Transcript length:", len(transcript))

In [ ]:
def split_scenes(text,words=35):

    w=text.split()
    scenes=[]

    for i in range(0,len(w),words):
        scenes.append(" ".join(w[i:i+words]))

    return scenes

scenes=split_scenes(transcript)

print("Total scenes:",len(scenes))
scenes[:5]

In [ ]:
import random

# ---------------------------------------------------
# Visual curiosity spikes (high retention triggers)
# ---------------------------------------------------

visual_spikes = [

# ---------- MONEY / POWER / SYSTEMS ----------
"massive global financial network visualized as glowing data streams flowing across the world map",
"traders staring at collapsing stock market screens inside a dark financial command center",
"giant vault door opening to reveal mountains of gold bars inside a secret underground bank",
"close-up of a powerful billionaire observing global financial markets on holographic displays",
"an enormous wall of digital currency transactions moving across a futuristic financial hub",

# ---------- SCIENCE / DISCOVERY ----------
"scientists staring in shock at a mysterious signal appearing on deep space telescope monitors",
"a glowing unknown object detected deep in space through a powerful observatory telescope",
"researchers discovering an unexpected anomaly inside a high-energy physics laboratory",
"microscopic view of unknown biological structures under a powerful electron microscope",
"a massive particle accelerator tunnel glowing with experimental energy",

# ---------- HEALTH / HUMAN BODY ----------
"dramatic medical visualization of the human brain lighting up with neural activity",
"scientists studying a mysterious biological process inside a futuristic research lab",
"ultra detailed microscopic view of cells dividing and repairing human tissue",
"researchers analyzing brain scans revealing unexpected neural patterns",
"a futuristic medical laboratory investigating the secrets of human aging",

# ---------- PSYCHOLOGY / HUMAN BEHAVIOR ----------
"intense close-up of a human face showing deep concentration and psychological tension",
"dark room filled with scientists observing human behavior through one-way glass",
"visualization of human thoughts as glowing neural pathways inside the brain",
"a psychologist analyzing subtle emotional reactions during an experiment",
"crowds of people moving in synchronized patterns revealing hidden behavioral trends",

# ---------- SUCCESS / PRODUCTIVITY ----------
"a focused individual working late at night surrounded by glowing data screens",
"dramatic cinematic shot of a person planning strategy on a giant wall of notes and diagrams",
"a powerful leader observing complex systems on massive command center displays",
"close-up of intense problem solving with equations and models covering a large glass wall",
"a person studying late under a single lamp surrounded by books and notes",

# ---------- STUDY / LEARNING ----------
"students intensely preparing for high-pressure exams inside a silent library",
"close-up of complex equations and diagrams being solved on a transparent digital screen",
"a classroom where a breakthrough moment suddenly changes understanding",
"a brain visualization showing knowledge connections forming rapidly",
"slow motion pages of scientific books revealing powerful ideas",

# ---------- TECHNOLOGY / FUTURE ----------
"a gigantic futuristic data center glowing with millions of servers",
"satellites orbiting earth transmitting enormous streams of data",
"an advanced robotics laboratory where machines are being developed",
"a futuristic control room monitoring global technology networks",
"a massive supercomputer processing enormous amounts of information",

# ---------- SURVIVAL / EXTREME CONDITIONS ----------
"a lone human standing against a massive storm in a dramatic survival scene",
"rescue teams navigating dangerous disaster zones under extreme conditions",
"dramatic survival moment in the middle of a frozen wilderness",
"intense close-up of a climber facing extreme danger high on a mountain",
"a massive natural disaster unfolding with humans struggling to survive",

# ---------- COSMIC / MYSTERY ----------
"deep space filled with mysterious cosmic structures never seen before",
"a strange signal pattern appearing on a space research control screen",
"a massive unknown cosmic phenomenon observed through powerful telescopes",
"scientists watching a mysterious astronomical event unfold",
"an unexplained object moving silently through deep space"
]

# ---------------------------------------------------
# Camera styles (prevents visual repetition)
# ---------------------------------------------------

camera_styles = [

# Establishing / scale
"wide cinematic establishing shot",
"epic wide landscape shot",
"aerial drone perspective",

# Subject emphasis
"extreme close up shot",
"macro detail lens shot",

# Dramatic perspective
"dramatic low angle shot",
"top down perspective",
"cinematic over the shoulder shot",

# Motion / dynamic
"tracking cinematic shot",
"dynamic cinematic angle",

# Film-style techniques
"shallow depth of field cinematic shot",
"dramatic silhouette lighting shot",
"backlit cinematic subject shot",
"moody atmospheric fog shot",
"high contrast noir style lighting",

# Documentary style
"handheld documentary camera shot",
"cinematic slow push-in shot",
"observational documentary style frame"
]

# ---------------------------------------------------
# Inject curiosity spikes every ~7 scenes
# (~28 seconds attention reset)
# ---------------------------------------------------

for i in range(0, len(scenes), 7):

    scenes[i] += " " + random.choice(visual_spikes)


# ---------------------------------------------------
# Cinematic prompt generator
# ---------------------------------------------------

def build_prompt(scene):

    camera = random.choice(camera_styles)

    return f"""
cinematic documentary film still

{camera}

IMAX cinema camera
anamorphic 35mm lens
shallow depth of field
captured mid motion

ultra photorealistic
extreme detail
8k resolution
HDR

dramatic cinematic lighting
volumetric light rays
deep shadows
high contrast

atmospheric depth
cinematic composition
epic scale environment

film grain
realistic textures
professional color grading

Netflix science documentary style
National Geographic cinematic photography

subject:
{scene}

mood:
mysterious
dramatic
thought provoking
"""


# ---------------------------------------------------
# Generate prompts
# ---------------------------------------------------

prompts = [build_prompt(s) for s in scenes]

prompts

In [ ]:
import requests

try:
    r = requests.get("https://huggingface.co")
    print("Internet working:", r.status_code)
except Exception as e:
    print("Internet error:", e)

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# -----------------------------------
# GPU speed optimization
# -----------------------------------

torch.backends.cuda.matmul.allow_tf32 = True

# -----------------------------------
# Load model
# -----------------------------------

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

pipe.enable_attention_slicing()
pipe.vae.enable_slicing()

# CPU offload reduces GPU memory spikes
pipe.enable_model_cpu_offload()

# -----------------------------------
# helper: trim prompts to CLIP limit
# -----------------------------------

def trim_prompt(prompt, max_words=120):
    words = prompt.split()
    return " ".join(words[:max_words])


# -----------------------------------
# generate images
# -----------------------------------

images = []

for i, p in enumerate(prompts):

    p = trim_prompt(p)

    with torch.autocast("cuda"):

        result = pipe(
            p,
            num_inference_steps=18,   # reduced from 28 (much faster)
            guidance_scale=8.5,
            height=720,
            width=1280,
            negative_prompt="nsfw, nude, porn, blurry, distorted, low quality"
        )

    img = result.images[0]

    path = f"{BASE}/scenes/scene_{i}.png"
    img.save(path)

    images.append(path)

    print(f"Generated scene {i+1}/{len(prompts)}")

images

In [ ]:
import cv2
import numpy as np
import os

# safety check
if "images" not in globals():
    raise Exception("Images list not found. Image generation step did not run.")

clips_folder = f"{BASE}/clips"
os.makedirs(clips_folder, exist_ok=True)

clips = []

for i, img_path in enumerate(images):

    img = cv2.imread(img_path)

    if img is None:
        print("Skipping missing image:", img_path)
        continue
    
    h, w, _ = img.shape

    out_path = f"{clips_folder}/clip_{i}.mp4"

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, 30, (w, h))

    for f in range(120):

        scale = 1 + (f * 0.002)

        M = cv2.getRotationMatrix2D((w/2, h/2), 0, scale)

        frame = cv2.warpAffine(img, M, (w, h))

        out.write(frame)

    out.release()

    clips.append(out_path)

clips[:5]

In [ ]:
list_file=f"{BASE}/clips_list.txt"

with open(list_file,"w") as f:

    for c in clips:
        f.write(f"file '{c}'\n")

!ffmpeg -y -f concat -safe 0 \
-i $BASE/clips_list.txt \
-c copy \
$BASE/video/visual.mp4

In [ ]:
!ffmpeg -y -i $BASE/video/visual.mp4 \
-vf "fade=t=in:st=0:d=1,fade=t=out:st=8:d=1" \
$BASE/video/visual_fx.mp4

In [ ]:
from gtts import gTTS
import os

os.makedirs(f"{BASE}/audio", exist_ok=True)

tts = gTTS(
    text=transcript,
    lang="en",
    slow=False,
    tld="com"
)

tts.save(f"{BASE}/audio/narration.wav")

print("Narration saved:", f"{BASE}/audio/narration.wav")

In [ ]:
import numpy as np
from scipy.io.wavfile import write
import os

os.makedirs(f"{BASE}/music", exist_ok=True)

sample_rate = 44100
duration = 20

t = np.linspace(0, duration, int(sample_rate * duration))

# simple cinematic pad chords
freqs = [220, 277, 329]  # A minor style chord

signal = sum(np.sin(2*np.pi*f*t) for f in freqs)

# normalize
signal = signal / np.max(np.abs(signal))

write(f"{BASE}/music/music.wav", sample_rate, signal.astype(np.float32))

print("Music generated:", f"{BASE}/music/music.wav")

In [ ]:
!ffmpeg -y \
-i $BASE/video/visual_fx.mp4 \
-i $BASE/audio/narration.wav \
-i $BASE/music/music.wav \
-filter_complex "[1:a][2:a]amix=inputs=2:duration=longest" \
-shortest \
$BASE/video/final_video.mp4

In [ ]:
import shutil

SOURCE = f"{BASE}/video/final_video.mp4"

OUTPUT = "/kaggle/working/final_video.mp4"

shutil.copy(
    SOURCE,
    OUTPUT
)

print("FINAL_VIDEO:", OUTPUT)